Let's explore how to provide image and audio input to our agents. 

**NOTE:** We will be encoding our image and audion input in base 64. which encodes binary (ones and zeros *a.k.a base 2*) tp 64 printable characters, or base 64.

In [1]:
#first import load_dotenv
from dotenv import load_dotenv

load_dotenv()

True

First, Let's create an image input in our jupyter notebook

In [2]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [3]:
print(uploader.value) # just to print the image details, verifying it has been uploaded and in which format.

({'name': 'food1.png', 'type': 'image/png', 'size': 124915, 'content': <memory at 0x0000015A98FAF4C0>, 'last_modified': datetime.datetime(2026, 1, 19, 14, 9, 33, 195000, tzinfo=datetime.timezone.utc)},)


Then we encode it in base 64.

In [4]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

Now let's create our standard agent (and model in our case)

In [2]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

agent = create_agent(model= model)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Then pass in a multimodal question to the agent.

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What is food is on this plate?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

On this plate, there is a variety of food items. I can identify:

*   **Eggs:** Several hard-boiled and soft-boiled eggs, cut in half.
*   **Tomatoes:** Cherry tomatoes and some chopped tomatoes.
*   **Fruit:** Sliced bananas and diced mango.
*   **Meat:** Slices of grilled or roasted meat, possibly pork or chicken.
*   **Nuts:** Walnuts.
*   **Vegetables:** Lettuce and sliced kiwi.

It appears to be a balanced meal with protein, carbohydrates, and fruits/vegetables.


For Audio input, the principle is the exact same.

*(Grab a base 64 encoding of an MP3 or .wav file and pass it to the agent along with a text question.)*

In [7]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.80it/s]

Done.


In [ ]:
agent = create_agent(
    model=model,
)

multimodal_question = HumanMessage(content=[
    #{"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"} #I had asked for a brief description of the Baroque era in art and music.
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The Baroque era in art music, spanning roughly from 1600 to 1750, was a period of dramatic contrasts, grandeur, and emotional intensity. It's characterized by:

*   **Ornamentation and Virtuosity:** Music became highly embellished with trills, runs, and other decorative elements, showcasing the performer's skill.
*   **Basso Continuo:** A foundational harmonic element, usually played by a keyboard instrument (like harpsichord or organ) and a bass instrument (like cello or bassoon), providing a continuous harmonic and rhythmic framework.
*   **Contrast and Dynamics:** Composers explored strong contrasts in volume (loud vs. soft), tempo, and instrumentation, often within the same piece.
*   **Development of New Forms:** Major instrumental forms like the concerto, sonata, and fugue emerged and flourished. Vocal forms like opera and oratorio also reached new heights.
*   **Doctrine of the Affections:** A belief that music could evoke specific emotions or "affections" in the listener, with 